In [ ]:
!pip install faker pandas

import sqlite3
import random
import pandas as pd
from datetime import datetime
from faker import Faker

# 1. SETUP & CONFIGURATION
fake = Faker()
Faker.seed(42)
random.seed(42)

DB_NAME = 'smartcart.db'
NUM_CUSTOMERS = 500
NUM_PRODUCTS = 500
NUM_ORDERS = 800

# Connect to SQLite Database
conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

# Enable Foreign Key Support in SQLite
cursor.execute("PRAGMA foreign_keys = ON;")

# 2. CREATE TABLES (Strict 3NF Schema)
tables_sql = [
    """CREATE TABLE IF NOT EXISTS Membership (
        loyalty_tier_id INTEGER PRIMARY KEY,
        tier VARCHAR(20) NOT NULL UNIQUE
    )""",

    """CREATE TABLE IF NOT EXISTS Category (
        category_id VARCHAR(10) PRIMARY KEY,
        category_name VARCHAR(50) NOT NULL UNIQUE
    )""",

    """CREATE TABLE IF NOT EXISTS Product (
        product_id VARCHAR(20) PRIMARY KEY,
        product_name VARCHAR(100) NOT NULL,
        unit_price REAL NOT NULL CHECK(unit_price >= 0),
        unit_cost REAL NOT NULL CHECK(unit_cost >= 0),
        category_id VARCHAR(10) NOT NULL,
        FOREIGN KEY(category_id) REFERENCES Category(category_id)
    )""",

    """CREATE TABLE IF NOT EXISTS Customer (
        customer_id VARCHAR(20) PRIMARY KEY,
        signup_date TEXT NOT NULL,
        age_band VARCHAR(15),
        address TEXT,
        city VARCHAR(50),
        email VARCHAR(100) UNIQUE NOT NULL,
        loyalty_tier_id INTEGER NOT NULL DEFAULT 1,
        FOREIGN KEY(loyalty_tier_id) REFERENCES Membership(loyalty_tier_id)
        -- Note: signup_date utilizes TEXT as SQLite stores dates as ISO 8601 strings
    )""",

    """CREATE TABLE IF NOT EXISTS Promotion (
        promo_id VARCHAR(20) PRIMARY KEY,
        promo_name VARCHAR(100) NOT NULL,
        promo_type VARCHAR(15) NOT NULL CHECK(promo_type IN ('PERCENT', 'FIXED')),
        discount_pct REAL DEFAULT NULL CHECK(discount_pct >= 0 AND discount_pct <= 1),
        discount_amount REAL DEFAULT NULL CHECK(discount_amount >= 0),
        loyalty_tier_id INTEGER,
        FOREIGN KEY(loyalty_tier_id) REFERENCES Membership(loyalty_tier_id)
    )""",

    """CREATE TABLE IF NOT EXISTS Promo_Category (
        promo_category_id INTEGER PRIMARY KEY AUTOINCREMENT,
        promo_id VARCHAR(20) NOT NULL,
        category_id VARCHAR(10) NOT NULL,
        FOREIGN KEY(promo_id) REFERENCES Promotion(promo_id),
        FOREIGN KEY(category_id) REFERENCES Category(category_id),
        UNIQUE(promo_id, category_id)
    )""",

    """CREATE TABLE IF NOT EXISTS Promo_Product (
        promo_product_id INTEGER PRIMARY KEY AUTOINCREMENT,
        promo_id VARCHAR(20) NOT NULL,
        product_id VARCHAR(20) NOT NULL,
        FOREIGN KEY(promo_id) REFERENCES Promotion(promo_id),
        FOREIGN KEY(product_id) REFERENCES Product(product_id),
        UNIQUE(promo_id, product_id)
    )""",

    """CREATE TABLE IF NOT EXISTS Orders (
        order_id VARCHAR(20) PRIMARY KEY,
        customer_id VARCHAR(20) NOT NULL,
        order_datetime TEXT NOT NULL,
        payment_method VARCHAR(30) NOT NULL,
        channel VARCHAR(20) NOT NULL DEFAULT 'Online',
        order_status VARCHAR(20) NOT NULL DEFAULT 'Completed',
        FOREIGN KEY(customer_id) REFERENCES Customer(customer_id)
    )""",

    """CREATE TABLE IF NOT EXISTS Order_Items (
        order_item_id INTEGER PRIMARY KEY AUTOINCREMENT,
        order_id VARCHAR(20) NOT NULL,
        product_id VARCHAR(20) NOT NULL,
        quantity INTEGER NOT NULL DEFAULT 1 CHECK(quantity > 0),
        unit_price_at_buy REAL NOT NULL CHECK(unit_price_at_buy >= 0),
        unit_cost_at_sale REAL NOT NULL CHECK(unit_cost_at_sale >= 0),
        discount_amount_applied REAL NOT NULL DEFAULT 0.00,
        promo_id VARCHAR(20) DEFAULT NULL,
        FOREIGN KEY(order_id) REFERENCES Orders(order_id),
        FOREIGN KEY(product_id) REFERENCES Product(product_id),
        FOREIGN KEY(promo_id) REFERENCES Promotion(promo_id),

        -- Enforce composite uniqueness to prevent duplicate product lines within a single order
        UNIQUE(order_id, product_id),

        -- Ensure applied discount does not exceed the total gross price of the line item
        CHECK(discount_amount_applied <= (unit_price_at_buy * quantity))
    )"""
]

# Execute DDL
for sql in tables_sql:
    cursor.execute(sql)

# 3. DATA GENERATORS (Business Logic & Constraints)

# Membership
memberships = [(1, 'Bronze'), (2, 'Silver'), (3, 'Gold')]
cursor.executemany("INSERT OR IGNORE INTO Membership VALUES (?, ?)", memberships)

# Category
categories = [('C1', 'Electronics'), ('C2', 'Skincare'), ('C3', 'Household Cleaning'), ('C4', 'Groceries'), ('C5', 'Clothing')]
cursor.executemany("INSERT OR IGNORE INTO Category VALUES (?, ?)", categories)

# Product (Gross Margin Target: 20-30%)
products_db = {}
for i in range(1, NUM_PRODUCTS + 1):
    pid = f"PRD{i:04d}"
    cat_id = random.choice(categories)[0]
    price = round(random.uniform(10.0, 500.0), 2)
    cost = round(price * random.uniform(0.70, 0.80), 2)
    pname = fake.word().capitalize() + " " + fake.word().capitalize()

    cursor.execute("INSERT OR IGNORE INTO Product VALUES (?, ?, ?, ?, ?)", (pid, pname, price, cost, cat_id))
    products_db[pid] = {'price': price, 'cost': cost, 'cat_id': cat_id}

# Customer
customers_db = {}
tiers, tier_weights = [1, 2, 3], [0.70, 0.20, 0.10]
for i in range(1, NUM_CUSTOMERS + 1):
    cid = f"CUS{i:04d}"
    tier = random.choices(tiers, weights=tier_weights, k=1)[0]
    # Ensure signup occurs prior to 2025 order period
    signup = fake.date_between(start_date=datetime(2023, 1, 1), end_date=datetime(2024, 12, 31))

    cursor.execute("INSERT OR IGNORE INTO Customer VALUES (?, ?, ?, ?, ?, ?, ?)",
                   (cid, signup.strftime("%Y-%m-%d"), random.choice(['18-25', '26-35', '36-45', '46+']),
                    fake.street_address(), fake.city(), fake.email(), tier))
    customers_db[cid] = {'tier': tier, 'signup': signup}

# Promotions & Mappings
promotions = [
    ('P1', 'Deep Clean Clearance - 60% Off', 'PERCENT', 0.60, None, None),
    ('P2', 'Gold Member Exclusive 20%', 'PERCENT', 0.20, None, 3),
    ('P3', 'Online Flash Sale 10 Off', 'FIXED', None, 10.00, None)
]
cursor.executemany("INSERT OR IGNORE INTO Promotion VALUES (?, ?, ?, ?, ?, ?)", promotions)

# Map P1 to Household Cleaning
cursor.execute("INSERT OR IGNORE INTO Promo_Category (promo_id, category_id) VALUES (?, ?)", ('P1', 'C3'))

# Map P3 to a random sample of products
p3_products = random.sample(list(products_db.keys()), 50)
for pid in p3_products:
    cursor.execute("INSERT OR IGNORE INTO Promo_Product (promo_id, product_id) VALUES (?, ?)", ('P3', pid))

# Orders and Order_Items (Shopping Cart Aggregation)
for i in range(1, NUM_ORDERS + 1):
    oid = f"ORD{i:05d}"
    cid = random.choice(list(customers_db.keys()))
    channel = random.choice(['Online', 'Offline'])

    # Orders placed within the 2025 fiscal year
    order_date = fake.date_time_between(start_date=datetime(2025, 1, 1), end_date=datetime(2025, 12, 31))

    cursor.execute("INSERT OR IGNORE INTO Orders VALUES (?, ?, ?, ?, ?, ?)",
                   (oid, cid, order_date.strftime("%Y-%m-%d %H:%M:%S"),
                    random.choice(['Credit Card', 'E-Wallet', 'Cash']), channel, 'Completed'))

    cust_tier = customers_db[cid]['tier']
    num_items = random.randint(1, 4)

    base_promo_chance = 0.20
    if cust_tier == 3: base_promo_chance = 0.85
    if channel == 'Online': base_promo_chance += 0.15

    # Dictionary to aggregate identical products before insertion
    cart = {}

    for _ in range(num_items):
        pid = random.choice(list(products_db.keys()))
        prod_info = products_db[pid]
        qty = random.randint(1, 3)
        unit_price = prod_info['price']
        unit_cost = prod_info['cost']

        applied_promo, discount_amount = None, 0.00

        # Promo assignment logic
        if random.random() <= base_promo_chance:
            eligible = []
            if prod_info['cat_id'] == 'C3': eligible.append('P1')
            if cust_tier == 3: eligible.append('P2')
            if pid in p3_products and channel == 'Online': eligible.append('P3')

            if eligible:
                if 'P1' in eligible and random.random() < 0.90:
                    applied_promo = 'P1'
                else:
                    applied_promo = random.choice(eligible)

                # Calculate total discount for the current product generation iteration
                total_line_price = unit_price * qty
                if applied_promo == 'P1':
                    discount_amount = round(total_line_price * 0.60, 2)
                elif applied_promo == 'P2':
                    discount_amount = round(total_line_price * 0.20, 2)
                elif applied_promo == 'P3':
                    discount_amount = 10.00 * qty
                    if discount_amount > total_line_price:
                        discount_amount = round(total_line_price - 1, 2)

        # Aggregate item quantities and discounts to prevent integrity constraint violations
        if pid in cart:
            cart[pid]['qty'] += qty
            cart[pid]['discount'] += (discount_amount if applied_promo else 0.00)
            if applied_promo:
                cart[pid]['promo_id'] = applied_promo
        else:
            cart[pid] = {
                'qty': qty,
                'price': unit_price,
                'cost': unit_cost,
                'promo_id': applied_promo,
                'discount': (discount_amount if applied_promo else 0.00)
            }

    # Insert aggregated cart contents into the database
    for pid_key, item in cart.items():
        cursor.execute("""
            INSERT OR IGNORE INTO Order_Items (order_id, product_id, quantity, unit_price_at_buy,
                                     unit_cost_at_sale, discount_amount_applied, promo_id)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (oid, pid_key, item['qty'], item['price'], item['cost'], round(item['discount'], 2), item['promo_id']))

# Commit transactions
conn.commit()

# 4. EXPORT TO CSV
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

for table_name in tables:
    table_name = table_name[0]
    df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    df.to_csv(f'{table_name}.csv', index=False)
    print(f"Exported: '{table_name}.csv' ({len(df)} records)")

conn.close()
print("\nDatabase initialization, synthetic data generation, and CSV extraction complete.")

Exported: 'Membership.csv' (3 records)
Exported: 'Category.csv' (5 records)
Exported: 'Product.csv' (500 records)
Exported: 'Customer.csv' (500 records)
Exported: 'Promotion.csv' (3 records)
Exported: 'Promo_Category.csv' (1 records)
Exported: 'sqlite_sequence.csv' (3 records)
Exported: 'Promo_Product.csv' (50 records)
Exported: 'Orders.csv' (800 records)
Exported: 'Order_Items.csv' (1984 records)

Database initialization, synthetic data generation, and CSV extraction complete.


In [ ]:
conn = sqlite3.connect('smartcart.db')

## **Gross Revenue**

In [ ]:
gross_revenue = """SELECT ROUND(SUM(unit_price_at_buy * quantity), 2) AS gross_revenue
FROM Order_Items"""
print(pd.read_sql_query(gross_revenue, conn))

   gross_revenue
0      978780.11


## **Net Profit**

In [ ]:
net_profit = """SELECT ROUND(SUM((unit_price_at_buy - unit_cost_at_sale) * quantity - discount_amount_applied), 2) AS net_profit
FROM Order_Items"""
print(pd.read_sql_query(net_profit, conn))

   net_profit
0   188265.46


## **Profit By Promotion**

In [ ]:
q1 = """SELECT COALESCE(p.promo_name, 'No Promotion') AS promotion, COUNT(*) AS items_sold,
    ROUND(SUM((oi.unit_price_at_buy - oi.unit_cost_at_sale) * oi.quantity - oi.discount_amount_applied), 2) AS total_profit
FROM Order_Items oi
LEFT JOIN Promotion p ON oi.promo_id = p.promo_id
GROUP BY promotion
ORDER BY total_profit DESC"""
df1 = pd.read_sql_query(q1, conn)
print("Profit by Promotion")
print(df1)

Profit by Promotion
                        promotion  items_sold  total_profit
0                    No Promotion        1674     202302.97
1        Online Flash Sale 10 Off          39       4374.21
2       Gold Member Exclusive 20%         150       3602.12
3  Deep Clean Clearance - 60% Off         121     -22013.84


## **Profit Margin by Category & Promotion**

In [ ]:
q2 = """SELECT c.category_name, COALESCE(p.promo_name, 'No Promo') AS promotion,
    ROUND(AVG(((oi.unit_price_at_buy - oi.unit_cost_at_sale) * oi.quantity - oi.discount_amount_applied) / (oi.unit_price_at_buy * oi.quantity) * 100), 2) AS avg_margin_percentage
FROM Order_Items oi
JOIN Product pr ON oi.product_id = pr.product_id
JOIN Category c ON pr.category_id = c.category_id
LEFT JOIN Promotion p ON oi.promo_id = p.promo_id
GROUP BY c.category_name, promotion"""
df2 = pd.read_sql_query(q2, conn)
print(df2)

         category_name                       promotion  avg_margin_percentage
0             Clothing       Gold Member Exclusive 20%                   4.90
1             Clothing                        No Promo                  25.21
2             Clothing        Online Flash Sale 10 Off                  19.84
3          Electronics       Gold Member Exclusive 20%                   4.61
4          Electronics                        No Promo                  24.75
5          Electronics        Online Flash Sale 10 Off                  18.06
6            Groceries       Gold Member Exclusive 20%                   4.23
7            Groceries                        No Promo                  24.43
8            Groceries        Online Flash Sale 10 Off                  19.96
9   Household Cleaning  Deep Clean Clearance - 60% Off                 -34.65
10  Household Cleaning       Gold Member Exclusive 20%                   6.44
11  Household Cleaning                        No Promo          

## **Promotional Effectiveness by Customer Loyalty**

In [ ]:
q3 = """SELECT m.tier, COALESCE(p.promo_name, 'No Promo') AS promotion,
    ROUND(SUM((oi.unit_price_at_buy - oi.unit_cost_at_sale) * oi.quantity - oi.discount_amount_applied), 2) AS total_profit,
    ROUND(AVG(((oi.unit_price_at_buy - oi.unit_cost_at_sale) * oi.quantity - oi.discount_amount_applied) / (oi.unit_price_at_buy * oi.quantity) * 100), 2) AS avg_margin_percentage
FROM Order_Items oi
JOIN Orders o ON oi.order_id = o.order_id
JOIN Customer cu ON o.customer_id = cu.customer_id
JOIN Membership m ON cu.loyalty_tier_id = m.loyalty_tier_id
LEFT JOIN Promotion p ON oi.promo_id = p.promo_id
GROUP BY m.tier, promotion"""

df3 = pd.read_sql_query(q3, conn)
print(df3)

     tier                       promotion  total_profit  avg_margin_percentage
0  Bronze  Deep Clean Clearance - 60% Off     -10454.83                 -35.16
1  Bronze                        No Promo     153143.54                  24.87
2  Bronze        Online Flash Sale 10 Off       2838.34                  19.85
3    Gold  Deep Clean Clearance - 60% Off      -6920.08                 -33.60
4    Gold       Gold Member Exclusive 20%       3602.12                   4.57
5    Gold                        No Promo       2200.01                  25.63
6    Gold        Online Flash Sale 10 Off        734.93                  16.96
7  Silver  Deep Clean Clearance - 60% Off      -4638.93                 -34.80
8  Silver                        No Promo      46959.42                  24.85
9  Silver        Online Flash Sale 10 Off        800.94                  18.90


## **Promotional Effectiveness by Sales Channel**

In [ ]:
q4 = """SELECT o.channel, COALESCE(p.promo_name, 'No Promotion') AS promotion,
    ROUND(SUM((oi.unit_price_at_buy - oi.unit_cost_at_sale) * oi.quantity - oi.discount_amount_applied), 2) AS total_profit,
    ROUND(AVG(((oi.unit_price_at_buy - oi.unit_cost_at_sale) * oi.quantity - oi.discount_amount_applied) / (oi.unit_price_at_buy * oi.quantity) * 100), 2) AS avg_margin_percent
FROM Order_Items oi
JOIN Orders o ON oi.order_id = o.order_id
LEFT JOIN Promotion p ON oi.promo_id = p.promo_id
GROUP BY o.channel, promotion
ORDER BY o.channel, total_profit DESC
"""
df4 = pd.read_sql_query(q4, conn)
print(df4)

   channel                       promotion  total_profit  avg_margin_percent
0  Offline                    No Promotion     111165.58               24.90
1  Offline       Gold Member Exclusive 20%       1644.94                4.62
2  Offline  Deep Clean Clearance - 60% Off     -10276.68              -34.54
3   Online                    No Promotion      91137.39               24.85
4   Online        Online Flash Sale 10 Off       4374.21               19.29
5   Online       Gold Member Exclusive 20%       1957.18                4.53
6   Online  Deep Clean Clearance - 60% Off     -11737.16              -34.74
